[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/06_%E7%99%BA%E5%B1%95_%E3%83%9E%E3%83%BC%E3%82%B1%E5%88%86%E6%9E%90/03_%E9%A1%A7%E5%AE%A2%E6%A7%8B%E6%88%90%E3%81%AE%E6%A4%9C%E5%AE%9A.ipynb)

In [ ]:
# === ① セットアップ（最初に実行してください）===
import pandas as pd               # 表データ
import numpy as np                # 数値計算
import matplotlib.pyplot as plt   # グラフ描画
import os
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の文字化け防止
# ローカルに data/ が無ければ公開リポジトリ(raw)から読み込む
RAW = 'https://raw.githubusercontent.com/Carlo-Broschi/statistics-python-for-students/main/data/'
def load(name):
    p = f'../data/{name}'
    return pd.read_csv(p) if os.path.exists(p) else pd.read_csv(RAW + name)
print('準備OK')
from scipy import stats

# 発展マーケ-03. 顧客構成の検定（適合度のカイ二乗）

> Colab（ブラウザ）で実行可。最初に「① セットアップ」セルを実行。
> **統計検定2級レベル**（実務でよく使う検定・分析）。scipy・statsmodels を使用。

**舞台設定**：顧客の業界構成は「均等」か、あるいは「想定したシェアどおり」か。観測した度数が想定の比率に合うかを**適合度のカイ二乗検定**で確かめます。

**使う統計（2級）**：適合度のカイ二乗検定・期待度数・自由度。

### 使うデータ：`btob_customers.csv`（顧客320件）
`業界`（6カテゴリ）の構成比を検定します。

## この章でできるようになること
- **適合度のカイ二乗検定**で「観測度数が想定比率に合うか」を検定できる
- 期待度数・自由度・p値を読める
- 適合度検定（1変数）と独立性検定（2変数）の違いを説明できる

> **前提**：カイ二乗・仮説検定（2級）　/　**所要**：約20分　/　**レベル**：2級

## 1. 業界の観測度数

In [ ]:
cust = load('btob_customers.csv')
obs = cust['業界'].value_counts()
print(obs)
obs.plot(kind='bar', figsize=(7,4), title='業界別の顧客数'); plt.ylabel('人数'); plt.xticks(rotation=0); plt.show()

## 2. 「6業界は均等か？」適合度検定

- H₀（帰無仮説）：6業界の顧客数は**均等**（各 1/6）
- H₁：均等でない

期待度数（均等なら各業界 320÷6 ≈ 53.3人）と観測度数のズレを測ります。

In [ ]:
k = len(obs)
exp_equal = [len(cust)/k]*k                       # 期待度数（均等）
chi2, p = stats.chisquare(obs.values, exp_equal)
print(f'自由度 = {k-1}')
print(f'カイ二乗 = {chi2:.2f},  p値 = {p:.4f}')
print('→ 5%で', ('均等でない' if p < 0.05 else '「均等でない」とは言えない（偏りの証拠なし）'))

## 3. 「想定シェアどおりか？」期待比率を指定して検定

営業の想定シェア（例：IT30%・金融20%・小売/医療各15%・製造/建設各10%）に合うかを検定します。

In [ ]:
plan = {'IT':0.30,'金融':0.20,'小売':0.15,'医療':0.15,'製造':0.10,'建設':0.10}
exp_plan = [len(cust)*plan[i] for i in obs.index]   # 想定シェアの期待度数
chi2b, pb = stats.chisquare(obs.values, exp_plan)
print(f'カイ二乗 = {chi2b:.2f},  p値 = {pb:.4f}')
print('→ 5%で', ('想定シェアと異なる' if pb < 0.05 else '想定どおりと言える'))

## 4. 適合度検定と独立性検定の違い

| | 適合度検定 | 独立性検定 |
|---|---|---|
| 対象 | **1変数**の分布 | **2変数**の関連 |
| 例 | 業界構成は想定どおり？ | 業界と受注は関係ある？ |
| 関数 | `chisquare` | `chi2_contingency` |

## まとめ
- **適合度のカイ二乗検定**は「観測度数が想定の比率に合うか」を検定する。
- 期待度数とのズレが大きいほどカイ二乗が大きく、有意になる。
- 1変数の分布は**適合度**、2変数の関連は**独立性**の検定を使う。

> **よくある間違い**
> - 期待度数が小さい（目安5未満）セルがあると検定の精度が落ちる。
> - 「有意でない」は「均等だと証明された」ではなく「偏りの証拠がない」。
> - 適合度（1変数）と独立性（2変数）を取り違えない。

## 確認テスト（自動採点）

`ans = None` を**自分の計算で置き換えて実行**すると、その場で正誤が出ます。

In [ ]:
# 採点用の関数（答え合わせに使うだけ・答えは表示しません）
def _check(name, got, want, tol=1e-2):
    if got is None:
        print(f'⬜ {name}: まだ入力されていません（ans を埋めてね）'); return
    ok = abs(got - want) <= tol
    print(('✅ 正解！ ' if ok else '❌ もう一度 ') + f'{name}: あなたの答え = {got}')

In [ ]:
# Q1: カテゴリが6つの適合度検定の自由度（カテゴリ数−1）を ans に
ans = None   # 例: 6-1
_check('Q1 自由度', ans, 6-1)

In [ ]:
# Q2: 320人を6業界に均等配分したときの期待度数を ans に
ans = None   # 例: 320/6
_check('Q2 期待度数', ans, 320/6)

In [ ]:
from scipy import stats
# Q3: 観測[30,30,40] 期待[33.3,33.3,33.3] の適合度カイ二乗のp値を ans に
ans = None   # 例: stats.chisquare([30,30,40],[100/3]*3).pvalue
_check('Q3 適合度p値', ans, stats.chisquare([30,30,40],[100/3]*3).pvalue)

---
## 練習問題

**問1.** 想定シェアを自分で変えて、結論（有意かどうか）がどう変わるか試そう。

**問2.** `業界` と `受注`… ではなく、別の質的列があれば独立性検定（`chi2_contingency`）も試そう。

**問3.**（考察）適合度検定で「有意でない」と出たとき、現場ではどう解釈・報告すべきか1〜2行で書こう。

> **解答例は別ページ**（ネタバレ防止）**[解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/06_%E7%99%BA%E5%B1%95_%E3%83%9E%E3%83%BC%E3%82%B1%E5%88%86%E6%9E%90/03_%E9%A1%A7%E5%AE%A2%E6%A7%8B%E6%88%90%E3%81%AE%E6%A4%9C%E5%AE%9A.md)**
>
> 自分で解いてから開きましょう。

## 用語集 ＆ チートシート

| 用語 | 意味 |
|---|---|
| 適合度検定 | 1変数の分布が想定比率に合うか |
| 期待度数 | 帰無仮説のもとで期待される度数 |
| 自由度 | カテゴリ数 − 1 |
| 独立性検定 | 2変数が無関係かを検定 |

```python
from scipy import stats
stats.chisquare(観測度数, 期待度数)   # 適合度検定
```